In [21]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [23]:
#
# load final X and y arrays from 'final_X_and_y.npz'
#

from pathlib import Path
import pickle

directory_path_input = Path("D:/ECG/input_for_model") # chose the right path beforhand
loaded_arrays = np.load(directory_path_input/'X_y_input.npz', allow_pickle=True)

# chose the file to restore:
X_train = loaded_arrays['a']
y_train = loaded_arrays['b']
X_val = loaded_arrays['c']
y_val = loaded_arrays['d']
X_test = loaded_arrays['e']
y_test = loaded_arrays['f']
y_test
loaded_arrays.close()

In [25]:
# ============================================================
# 0) Configuration
# ============================================================

#SEED = 42
SEED = 31
#SEED = 7

tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


OUTPUT_DIR = Path("gru_4_class_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


model_config = {
    "input_shape": (1000, 1),   # adjust to current ECG sequence length

    # GRU architecture
    "gru_units_1": 128,
    "gru_units_2": 64,
    "bidirectional": True,
    "dropout": 0.2,
    "head_units": 64,

    # Classification task
    "num_classes": 4,

    # Training
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 30,
    "early_stopping_patience": 5,
    "random_seed": SEED
}


class_weight = {0: 0.526009657594381,
 1: 1.3992798754379137,
 2: 1.0189200680272108,
 3: 2.4825621546961325}


In [27]:

# ============================================================
# 1) Prepare labels
# ============================================================

def prepare_multiclass_labels(y, num_classes):
    """
    Convert multiclass labels to a one-dimensional array
    containing sparse integer labels.

    Supported inputs:
    - shape (N,)
    - shape (N, 1)
    - one-hot encoded shape (N, num_classes)
    """
    y = np.asarray(y)

    # One-hot encoded labels
    if y.ndim == 2 and y.shape[1] == num_classes:
        y = np.argmax(y, axis=1)

    # Column vector
    elif y.ndim == 2 and y.shape[1] == 1:
        y = y.ravel()

    elif y.ndim != 1:
        raise ValueError(
            "Multiclass labels must have shape "
            "(N,), (N, 1), or (N, num_classes). "
            f"Received shape: {y.shape}"
        )

    y = y.astype(np.int32)

    unique_labels = np.unique(y)

    expected_labels = np.arange(num_classes)

    if not np.all(np.isin(unique_labels, expected_labels)):
        raise ValueError(
            f"Labels must contain only integers from "
            f"0 to {num_classes - 1}. "
            f"Found labels: {unique_labels}"
        )

    return y


# ------------------------------------------------------------
# Prepare labels
# ------------------------------------------------------------

y_train_mc = prepare_multiclass_labels(
    y_train,
    model_config["num_classes"]
)

y_val_mc = prepare_multiclass_labels(
    y_val,
    model_config["num_classes"]
)

y_test_mc = prepare_multiclass_labels(
    y_test,
    model_config["num_classes"]
)


# ------------------------------------------------------------
# Prepare ECG input data
# ------------------------------------------------------------

X_train_model = np.asarray(
    X_train,
    dtype=np.float32
)

X_val_model = np.asarray(
    X_val,
    dtype=np.float32
)

X_test_model = np.asarray(
    X_test,
    dtype=np.float32
)


# Add feature/channel dimension if data have shape:
# (samples, timesteps)
#
# GRU expects:
# (samples, timesteps, features)

if X_train_model.ndim == 2:
    X_train_model = X_train_model[..., np.newaxis]

if X_val_model.ndim == 2:
    X_val_model = X_val_model[..., np.newaxis]

if X_test_model.ndim == 2:
    X_test_model = X_test_model[..., np.newaxis]


# ------------------------------------------------------------
# Check input shapes
# ------------------------------------------------------------

if X_train_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_train has model input shape {X_train_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )

if X_val_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_val has model input shape {X_val_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )

if X_test_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_test has model input shape {X_test_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )


# ------------------------------------------------------------
# Check numbers of samples and labels
# ------------------------------------------------------------

if len(X_train_model) != len(y_train_mc):
    raise ValueError(
        f"Training data mismatch: "
        f"{len(X_train_model)} samples but "
        f"{len(y_train_mc)} labels."
    )

if len(X_val_model) != len(y_val_mc):
    raise ValueError(
        f"Validation data mismatch: "
        f"{len(X_val_model)} samples but "
        f"{len(y_val_mc)} labels."
    )

if len(X_test_model) != len(y_test_mc):
    raise ValueError(
        f"Test data mismatch: "
        f"{len(X_test_model)} samples but "
        f"{len(y_test_mc)} labels."
    )


In [29]:

# ============================================================
# 3) GRU model builder
# ============================================================

def build_multiclass_gru(
    input_shape=(1000, 1),
    gru_units_1=128,
    gru_units_2=64,
    bidirectional=True,
    dropout=0.2,
    head_units=64,
    num_classes=4
):
    inputs = tf.keras.Input(
        shape=input_shape,
        name="ecg_input"
    )

    x = inputs

    # --------------------------------------------------------
    # First GRU layer
    # --------------------------------------------------------
    gru_1 = tf.keras.layers.GRU(
        units=gru_units_1,
        return_sequences=True,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        bias_initializer="zeros",
        name="gru_1"
    )

    if bidirectional:
        x = tf.keras.layers.Bidirectional(
            gru_1,
            name="bidirectional_gru_1"
        )(x)
    else:
        x = gru_1(x)

    if dropout > 0:
        x = tf.keras.layers.Dropout(
            rate=dropout,
            name="dropout_1"
        )(x)

    # --------------------------------------------------------
    # Second GRU layer
    # --------------------------------------------------------
    gru_2 = tf.keras.layers.GRU(
        units=gru_units_2,
        return_sequences=False,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        bias_initializer="zeros",
        name="gru_2"
    )

    if bidirectional:
        x = tf.keras.layers.Bidirectional(
            gru_2,
            name="bidirectional_gru_2"
        )(x)
    else:
        x = gru_2(x)

    if dropout > 0:
        x = tf.keras.layers.Dropout(
            rate=dropout,
            name="dropout_2"
        )(x)

    # --------------------------------------------------------
    # Optional Dense head
    # --------------------------------------------------------
    if head_units > 0:
        x = tf.keras.layers.Dense(
            units=head_units,
            activation="relu",
            kernel_initializer="he_normal",
            name="dense_head"
        )(x)

    # --------------------------------------------------------
    # Multiclass output
    # --------------------------------------------------------
    outputs = tf.keras.layers.Dense(
        units=num_classes,
        activation="softmax",
        name="multiclass_output"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="multiclass_ecg_gru"
    )

    return model


model = build_multiclass_gru(
    input_shape=model_config["input_shape"],
    gru_units_1=model_config["gru_units_1"],
    gru_units_2=model_config["gru_units_2"],
    bidirectional=model_config["bidirectional"],
    dropout=model_config["dropout"],
    head_units=model_config["head_units"],
    num_classes=model_config["num_classes"]
)

In [31]:
# ============================================================
# 3) Compile model
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=model_config["learning_rate"]
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

# ============================================================
# 4) Store architecture and model summary
# ============================================================

with open(
    OUTPUT_DIR / "model_configuration.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_config,
        file,
        indent=4,
        default=str
    )


with open(
    OUTPUT_DIR / "model_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    model.summary(
        print_fn=lambda line: file.write(line + "\n")
    )


# Store the complete Keras model architecture as JSON.
with open(
    OUTPUT_DIR / "model_architecture.json",
    "w",
    encoding="utf-8"
) as file:
    file.write(model.to_json())



In [35]:
# ============================================================
# 5) Callbacks
# ============================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=model_config["early_stopping_patience"],
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=OUTPUT_DIR / "best_4_class_gru.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=OUTPUT_DIR / "training_log.csv",
        separator=",",
        append=False
    ),

    tf.keras.callbacks.TerminateOnNaN()
]


In [37]:
# ============================================================
# 6) Train model
# ============================================================

history = model.fit(
    X_train_model,
    y_train_mc,
    validation_data=(
        X_val_model,
        y_val_mc
    ),
    epochs=model_config["max_epochs"],
    batch_size=model_config["batch_size"],
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)


# ============================================================
# 7) Store training history
# ============================================================

history_df = pd.DataFrame(history.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"

history_df.to_csv(
    OUTPUT_DIR / "training_history.csv"
)


# ============================================================
# 8) Generate class predictions
# ============================================================

train_probabilities = model.predict(
    X_train_model,
    batch_size=model_config["batch_size"],
    verbose=0
)

val_probabilities = model.predict(
    X_val_model,
    batch_size=model_config["batch_size"],
    verbose=0
)

test_probabilities = model.predict(
    X_test_model,
    batch_size=model_config["batch_size"],
    verbose=0
)


train_predictions = np.argmax(
    train_probabilities,
    axis=1
).astype(np.int32)

val_predictions = np.argmax(
    val_probabilities,
    axis=1
).astype(np.int32)

test_predictions = np.argmax(
    test_probabilities,
    axis=1
).astype(np.int32)

# ============================================================
# 9) Calculate final metrics
# ============================================================

train_loss, train_keras_accuracy = model.evaluate(
    X_train_model,
    y_train_mc,
    batch_size=model_config["batch_size"],
    verbose=0
)

val_loss, val_keras_accuracy = model.evaluate(
    X_val_model,
    y_val_mc,
    batch_size=model_config["batch_size"],
    verbose=0
)

test_loss, test_keras_accuracy = model.evaluate(
    X_test_model,
    y_test_mc,
    batch_size=model_config["batch_size"],
    verbose=0
)


train_accuracy = accuracy_score(
    y_train_mc,
    train_predictions
)

val_accuracy = accuracy_score(
    y_val_mc,
    val_predictions
)

test_accuracy = accuracy_score(
    y_test_mc,
    test_predictions
)


train_f1 = f1_score(
    y_train_mc,
    train_predictions,
    average="macro",
    zero_division=0
)

val_f1 = f1_score(
    y_val_mc,
    val_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_test_mc,
    test_predictions,
    average="macro",
    zero_division=0
)


train_confusion_matrix = confusion_matrix(
    y_train_mc,
    train_predictions,
    labels=[0, 1, 2, 3]
)


val_confusion_matrix = confusion_matrix(
    y_val_mc,
    val_predictions,
    labels=[0, 1, 2, 3]
)

test_confusion_matrix = confusion_matrix(
    y_test_mc,
    test_predictions,
    labels=[0, 1, 2, 3]
)

# ============================================================
# 10) Store final metrics
# ============================================================

final_metrics = {
    "num_classes": model_config["num_classes"],
    "number_of_model_parameters": int(model.count_params()),

    "training": {
        "loss": float(train_loss),
        "accuracy": float(train_accuracy),
        "keras_accuracy": float(train_keras_accuracy),
        "macro_f1_score": float(train_f1)
    },

    "validation": {
        "loss": float(val_loss),
        "accuracy": float(val_accuracy),
        "keras_accuracy": float(val_keras_accuracy),
        "macro_f1_score": float(val_f1)
    },
    "test": {
        "loss": float(test_loss),
        "accuracy": float(test_accuracy),
        "keras_accuracy": float(test_keras_accuracy),
        "macro_f1_score": float(test_f1)
    }   
}


with open(
    OUTPUT_DIR / "final_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metrics,
        file,
        indent=4
    )


# Also store the main metrics in table form.
metrics_df = pd.DataFrame(
    {
        "dataset": ["training", "validation", "test"],
        "loss": [train_loss, val_loss, test_loss],
        "accuracy": [train_accuracy, val_accuracy, test_accuracy],
        "macro_f1_score": [train_f1, val_f1, test_f1]
    }
)

metrics_df.to_csv(
    OUTPUT_DIR / "final_metrics.csv",
    index=False
)


# ============================================================
# 11) Store confusion matrices
# ============================================================

confusion_labels = [
    "predicted_class_0",
    "predicted_class_1",
    "predicted_class_2",
    "predicted_class_3"
]

confusion_index = [
    "true_class_0",
    "true_class_1",
    "true_class_2",
    "true_class_3"
]


train_cm_df = pd.DataFrame(
    train_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)

val_cm_df = pd.DataFrame(
    val_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)

test_cm_df = pd.DataFrame(
    test_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)

train_cm_df.to_csv(
    OUTPUT_DIR / "training_confusion_matrix.csv"
)

val_cm_df.to_csv(
    OUTPUT_DIR / "validation_confusion_matrix.csv"
)

test_cm_df.to_csv(
    OUTPUT_DIR / "test_confusion_matrix.csv"
)

# ============================================================
# 12) Store detailed classification reports
# ============================================================

train_report = classification_report(
    y_train_mc,
    train_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "class_0",
        "class_1",
        "class_2",
        "class_3"
    ],
    output_dict=True,
    zero_division=0
)

val_report = classification_report(
    y_val_mc,
    val_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "class_0",
        "class_1",
        "class_2",
        "class_3"
    ],
    output_dict=True,
    zero_division=0
)

test_report = classification_report(
    y_test_mc,
    test_predictions,
    labels=[0, 1, 2, 3],
    target_names=[
        "class_0",
        "class_1",
        "class_2",
        "class_3"
    ],
    output_dict=True,
    zero_division=0
)

train_report_df = pd.DataFrame(train_report).transpose()
val_report_df = pd.DataFrame(val_report).transpose()

test_report_df = pd.DataFrame(test_report).transpose()


train_report_df.to_csv(
    OUTPUT_DIR / "training_classification_report.csv"
)

val_report_df.to_csv(
    OUTPUT_DIR / "validation_classification_report.csv"
)

test_report_df.to_csv(
    OUTPUT_DIR / "test_classification_report.csv"
)


# ============================================================
# 13) Store predictions and probabilities
# ============================================================

train_prediction_data = {
    "true_label": y_train_mc,
    "predicted_label": train_predictions
}

val_prediction_data = {
    "true_label": y_val_mc,
    "predicted_label": val_predictions
}

test_prediction_data = {
    "true_label": y_test_mc,
    "predicted_label": test_predictions
}

for class_index in range(model_config["num_classes"]):
    train_prediction_data[
        f"probability_class_{class_index}"
    ] = train_probabilities[:, class_index]

    val_prediction_data[
        f"probability_class_{class_index}"
    ] = val_probabilities[:, class_index]

    test_prediction_data[
        f"probability_class_{class_index}"
    ] = test_probabilities[:, class_index]

train_prediction_df = pd.DataFrame(train_prediction_data)
val_prediction_df = pd.DataFrame(val_prediction_data)
test_prediction_df = pd.DataFrame(test_prediction_data)


train_prediction_df.to_csv(
    OUTPUT_DIR / "training_predictions.csv",
    index=False
)

val_prediction_df.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)

test_prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

# ============================================================
# 14) Save final model
# ============================================================

model.save(
    OUTPUT_DIR / "final_4_class_gru.keras"
)


# ============================================================
# 15) Print summary
# ============================================================

print("\nTraining completed.")
print(f"Results directory: {OUTPUT_DIR.resolve()}")
print(f"Number of parameters: {model.count_params():,}")

print("\nTraining metrics:")
print(f"Loss:     {train_loss:.4f}")
print(f"Accuracy: {train_accuracy:.4f}")
print(f"F1-score: {train_f1:.4f}")
print("Confusion matrix:")
print(train_confusion_matrix)

print("\nTraining classification report:")
print(train_report_df)

print("\nValidation metrics:")
print(f"Loss:     {val_loss:.4f}")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"F1-score: {val_f1:.4f}")
print("Confusion matrix:")
print(val_confusion_matrix)

print("\nValidation classification report:")
print(val_report_df)

print("\nTest metrics:")
print(f"Loss:     {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1-score: {test_f1:.4f}")
print("Confusion matrix:")
print(test_confusion_matrix)

print("\nTest classification report:")
print(test_report_df)



Epoch 1/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.2282 - loss: 1.3922
Epoch 1: val_loss improved from None to 1.38481, saving model to gru_4_class_results\best_4_class_gru.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 1832s 8s/step - accuracy: 0.2413 - loss: 1.3854 - val_accuracy: 0.2381 - val_loss: 1.3848
Epoch 2/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.2442 - loss: 1.3914
Epoch 2: val_loss improved from 1.38481 to 1.37810, saving model to gru_4_class_results\best_4_class_gru.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 1859s 8s/step - accuracy: 0.2527 - loss: 1.3841 - val_accuracy: 0.2707 - val_loss: 1.3781
Epoch 3/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.2931 - loss: 1.3908
Epoch 3: val_loss did not improve from 1.37810
225/225 ━━━━━━━━━━━━━━━━━━━━ 1801s 8s/step - accuracy: 0.2764 - loss: 1.3832 - val_accuracy: 0.2577 - val_loss: 1.3821
Epoch 4/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2893 - loss: 1.3904
Epoch 4: val_loss did not improve fro